# Exploration de l'API TED (marchés publics européens)

**Projet :** Plateforme d'automatisation de la prospection (veille AO / RFP / CFP)
**Module :** 1 - Collecte
**Source :** TED (Tenders Electronic Daily), le journal officiel des marchés publics de l'UE.

## Pourquoi TED

La BCEAO publie très peu de cybersécurité. Je cherche une source avec du volume et de vraies
opportunités cyber. TED est une **API officielle** : données structurées (pas de HTML à scraper),
énorme volume (toute l'Europe), filtrage par mots-clés / pays / catégorie.

## Ma démarche

Je ne connais pas cette API. Je la découvre **par tâtonnements** : je tente un appel, je lis ce que
l'API répond (surtout ses erreurs, très instructives), et j'ajuste. Je veux qu'aucun nom de champ
n'apparaisse "par magie" : chaque nom que j'utilise, je montre d'abord d'où il vient.

## 1. Est-ce que l'API répond ?

Je fais l'appel le plus simple possible et je regarde le **code HTTP** (200 = OK, 4xx = j'ai fait
une erreur, 5xx = souci serveur).

In [1]:
import requests
import json
from collections import Counter
from datetime import date

URL = "https://api.ted.europa.eu/v3/notices/search"

r = requests.post(URL, json={"query": "publication-date>=today(-1)"})
print("Code HTTP :", r.status_code)

Code HTTP : 400


**400** : l'API n'est pas contente. Ce n'est pas un échec, c'est une piste : elle va me dire
ce qui manque. Je lis le message.

In [2]:
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

{
  "message": "Validation error",
  "error": [
    {
      "objectName": "publicExpertSearchRequestV1",
      "field": "fields",
      "message": "must not be empty"
    }
  ]
}


Le message dit : le champ **`fields` ne doit pas être vide**. L'API veut que je précise quelles
informations je veux pour chaque annonce. Mais je ne connais aucun nom de champ valide... Je vais
en tenter un au hasard pour voir comment l'API réagit.

## 2. Découvrir les noms de champs (grâce à l'erreur)

Je tente un nom intuitif au hasard : `"title"`.

In [3]:
r = requests.post(URL, json={"query": "publication-date>=today(-1)", "fields": ["title"]})
print("Code HTTP :", r.status_code)
print(json.dumps(r.json(), ensure_ascii=False)[:350])

Code HTTP : 400
{"message": "Parameter 'fields' contains unsupported value (supported values are: sme-part,touchpoint-gateway-ted-esen,submission-url-lot,organisation-person-addinfo-part,no-negocaition-necessary-lot,BT-13(t)-Part,organisation-city-serv-prov,result-framework-maximum-value-cur-notice,BT-821-Lot,touchpoint-partname-tenderer,touchpoint-internet-addres


Encore 400, mais le message est en or : « unsupported value (**supported values are:** ... ) »
suivi d'une longue liste. En me trompant, l'API m'a **listé tous les champs valides**. Je récupère
cette liste et je la compte.

In [4]:
message = r.json()["message"]
liste_brute = message.split("supported values are:")[1].rstrip(")").strip()
champs = [c.strip() for c in liste_brute.split(",") if c.strip()]
print("Nombre de champs proposés par l'API :", len(champs))
print("Exemples (10 premiers) :", champs[:5], "...")

Nombre de champs proposés par l'API : 1830
Exemples (10 premiers) : ['sme-part', 'touchpoint-gateway-ted-esen', 'submission-url-lot', 'organisation-person-addinfo-part', 'no-negocaition-necessary-lot'] ...


**Attention :** ces 1830, ce sont des **champs** (les colonnes possibles : titre, date, pays...),
PAS le nombre d'annonces. Je verrai le nombre d'annonces plus loin. Pour l'instant, je cherche dans
cette liste les champs dont j'ai besoin.

## 3. Chercher mes champs dans la liste (avant de les utiliser)

Je ne veux pas utiliser un nom de champ sans l'avoir vu dans la liste. Donc je fouille la liste avec
des mots-clés correspondant aux informations utiles : identifiant, titre, date, pays, lien, catégorie.
J'écarte les champs préfixés `BT-` (codes techniques peu lisibles) juste pour la lisibilité.

In [5]:
for mot in ["publication", "number", "title", "deadline", "buyer-country", "buyer-name", "links", "cpv"]:
    trouves = [c for c in champs if mot in c.lower() and not c.startswith("BT-")][:5]
    print(f"{mot:16} -> {trouves}")

publication      -> ['publication-date', 'publication-number', 'publication-name', 'notice-preferred-publication-date']
number           -> ['award-criterion-number-fixed-glo', 'framework-maximum-participants-number-lot', 'number-tender-applications-ipi-measure-res', 'OPA-36-Part-Number', 'award-criterion-number-fixed-lot']
title            -> ['review-title', 'announcement-title', 'contract-title', 'notice-title', 'title-part']
deadline         -> ['deadline-receipt-tender-date-lot', 'tender-validity-deadline-unit-lot', 'deadline-time-part', 'deadline-receipt-expressions-time-lot', 'deadline-receipt-request']
buyer-country    -> ['buyer-country-sub', 'buyer-country']
buyer-name       -> ['buyer-name']
links            -> ['links']
cpv              -> ['classification-cpv']


Maintenant je vois d'où viennent les noms que je vais utiliser. En particulier,
**`publication-number`** apparaît bien dans la liste (sous "publication" et "number") : c'est
l'identifiant de l'annonce. Je retiens : `publication-number`, `notice-title`, `buyer-name`,
`buyer-country`, `publication-date`, `deadline`, `classification-cpv`, `links`.

## 4. Premier appel qui marche, et surtout : que renvoie l'API ?

J'utilise `publication-number` (trouvé dans la liste) pour faire un appel valide. Puis, au lieu de
supposer la forme de la réponse, je **regarde les clés** que l'API me renvoie.

In [6]:
r = requests.post(URL, json={"query": "publication-date>=today(-1)",
                              "fields": ["publication-number"], "limit": 3})
reponse = r.json()
print("Code HTTP :", r.status_code)
print("\nLes CLÉS que l'API me renvoie :")
for cle, val in reponse.items():
    apercu = f"liste de {len(val)} éléments" if isinstance(val, list) else val
    print(f"   {cle} = {apercu}")

Code HTTP : 200

Les CLÉS que l'API me renvoie :
   notices = liste de 3 éléments
   totalNoticeCount = 7321
   iterationNextToken = None
   timedOut = False


Voilà la structure de la réponse, sans rien deviner :
- **`notices`** : la liste des annonces demandées (ici 3, à cause de `limit=3`) ;
- **`totalNoticeCount`** : le nombre **total** d'annonces correspondant à ma recherche ;
- **`iterationNextToken`** : un jeton pour récupérer la suite (pagination) ;
- **`timedOut`** : si la requête a expiré.

C'est ici que je découvre **`totalNoticeCount`** : ce n'est pas un champ que je demande, c'est une
clé que l'API met dans sa réponse. C'est lui qui me donne le nombre de **lignes** (annonces).

## 5. Combien d'annonces existent ? (lignes, pas champs)

Je me fais une petite fonction pour lire `totalNoticeCount` selon une recherche.

In [7]:
def total_annonces(query, scope="ACTIVE"):
    body = {"query": query, "fields": ["publication-number"], "limit": 1, "scope": scope}
    return requests.post(URL, json=body).json().get("totalNoticeCount")

print("Annonces publiées depuis 1 jour  :", total_annonces("publication-date>=today(-1)"))
print("Annonces publiées depuis 7 jours :", total_annonces("publication-date>=today(-7)"))

Annonces publiées depuis 1 jour  : 7320
Annonces publiées depuis 7 jours : 21952


Plusieurs milliers par jour. Sur le **site** TED, sans filtre, j'obtiens ~6 millions. Pourquoi
l'écart ? Parce que le site montre **tout l'historique**, alors que je filtre sur les derniers jours.
Je le vérifie en enlevant le filtre de date (scope ALL).

In [8]:
def total_all(query):
    body = {"query": query, "fields": ["publication-number"], "limit": 1, "scope": "ALL"}
    return requests.post(URL, json=body).json().get("totalNoticeCount")

print("Tout l'historique :", total_all("publication-date>=today(-9000)"))
print("Depuis 1 an       :", total_all("publication-date>=today(-365)"))
print("Depuis 1 jour     :", total_all("publication-date>=today(-1)"))

Tout l'historique : 6937929
Depuis 1 an       : 900320
Depuis 1 jour     : 7321


Confirmé : tout l'historique donne ~6-7 millions, comme le site. L'API et le site sont donc
cohérents. **Conséquence :** pour ma veille, je filtrerai toujours par date récente + cyber + scope
ACTIVE, et je ferai tourner la collecte chaque jour (sinon le volume est ingérable).

## 6. Une fonction de recherche réutilisable

In [9]:
CHAMPS = ["publication-number", "notice-title", "buyer-name", "buyer-country",
          "publication-date", "deadline", "notice-type", "classification-cpv", "links"]

def chercher(query, limit=50, scope="ACTIVE"):
    body = {"query": query, "fields": CHAMPS, "limit": limit, "page": 1, "scope": scope}
    r = requests.post(URL, json=body, timeout=40)
    r.raise_for_status()
    return r.json()

## 7. Regarder une annonce en détail

Avant de filtrer, je regarde la structure brute d'une annonce (c'est là qu'on trouve les surprises).

In [10]:
data = chercher("publication-date>=today(-2)", limit=5)
print(json.dumps(data["notices"][0], indent=2, ensure_ascii=False)[:1000])

{
  "notice-type": "cn-standard",
  "publication-number": "413513-2026",
  "classification-cpv": [
    "60112000",
    "60112000"
  ],
  "buyer-name": {
    "deu": [
      "Zweckverband Verkehrsverbund Bremen/Niedersachsen (ZVBN)"
    ]
  },
  "publication-date": "2026-06-17+02:00",
  "links": {
    "xml": {
      "MUL": "https://ted.europa.eu/en/notice/413513-2026/xml"
    },
    "pdf": {
      "BUL": "https://ted.europa.eu/bg/notice/413513-2026/pdf",
      "SPA": "https://ted.europa.eu/es/notice/413513-2026/pdf",
      "CES": "https://ted.europa.eu/cs/notice/413513-2026/pdf",
      "DAN": "https://ted.europa.eu/da/notice/413513-2026/pdf",
      "DEU": "https://ted.europa.eu/de/notice/413513-2026/pdf",
      "EST": "https://ted.europa.eu/et/notice/413513-2026/pdf",
      "ELL": "https://ted.europa.eu/el/notice/413513-2026/pdf",
      "ENG": "https://ted.europa.eu/en/notice/413513-2026/pdf",
      "FRA": "https://ted.europa.eu/fr/notice/413513-2026/pdf",
      "GLE": "https://ted.europ

J'observe trois formats à gérer : le **titre** est un dictionnaire par langue, les **dates** ont
un fuseau (ex. `2026-06-18+02:00`), et `classification-cpv` est une **liste** de codes.

## 8. Isoler la cybersécurité : la nomenclature CPV officielle

Sur le site, à gauche, des filtres regroupent les annonces par grandes catégories (« Computer and
Related Services »...). En cliquant, j'ai vu qu'elles correspondent à des **codes CPV**. Mais le site
affiche aussi des **nombres** d'annonces par catégorie, et je ne veux PAS identifier mes codes à
partir de ces nombres : ils changent chaque jour (de nouvelles annonces arrivent), donc ce ne serait
pas reproductible.

La source **rigoureuse et stable**, c'est la **nomenclature CPV officielle** de l'UE : un fichier qui
associe chaque code à son libellé, et qui ne change pas dans le temps (version 2008, toujours en
vigueur). Je l'ai téléchargé une fois depuis TED/SIMAP et rangé dans mon projet. Je le charge.

In [19]:
import pandas as pd

# Fichier officiel CPV téléchargé depuis TED/SIMAP (adapte le chemin si besoin)
CHEMIN_CPV = "../data/cpv_2008.xlsx"
cpv = pd.read_excel(CHEMIN_CPV)
print("Dimensions (codes, langues) :", cpv.shape)
print("Colonnes (langues) :", list(cpv.columns)[:12], "...")
cpv.head(3)

Dimensions (codes, langues) : (9454, 25)
Colonnes (langues) : ['CODE', 'BG', 'CS', 'DA', 'DE', 'EL', 'EN', 'ES', 'ET', 'FI', 'FR', 'GA'] ...


c:\Users\ThinkPad\Desktop\Borji Emna\veille-ao-pwnpatch\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


,CODE,BG,CS,DA,DE,EL,EN,ES,ET,FI,...,LT,LV,MT,NL,PL,PT,RO,SK,SL,SV
0,03000000-1,"Продукти на земеделието, животновъдството, риб...","Produkty zemědělství, hospodářské produkty, pr...","Produkter fra landbrug, husdyrbrug, fiskeri og...",Landwirtschaftliche Erzeugnisse des Pflanzenba...,"Προϊόντα γεωργίας, κτηνοτροφίας, αλιείας, δασο...","Agricultural, farming, fishing, forestry and r...","Productos de la agricultura, ganadería, pesca,...","Põllumajandussaadused, loomakasvatus-, kalandu...","Maa-, karja-, kala- ja metsätaloustuotteet sek...",...,"Žemės ūkio, ūkininkavimo, žvejybos, miškininky...","Lauksaimniecības, saimniecības, zivsaimniecība...","Prodotti agrikoli, tal-biedja, tas-sajd, tal-f...","Landbouw- en veeteelt-, kwekerij-, visserij-, ...","Produkty rolnictwa, hodowli, rybołówstwa, leśn...","Produtos da agricultura, da pesca, da silvicul...","Produse agricole, de fermă, de pescuit, de sil...","Poľnohospodárske, farmárske, rybárske, lesníck...","Kmetijski, ribiški, gozdarski in z njimi povez...","Jordbruks-, jakt-, fiske- och skogsbruksproduk..."
1,03100000-2,Продукти на земеделието и градинарството,Produkty zemědělství a zahradnictví,Produkter fra land- og havebrug,Landwirtschafts- und Gartenbauerzeugnisse,Προϊόντα γεωργίας και κηπευτικής,Agricultural and horticultural products,Productos de la agricultura y horticultura,Põllumajandussaadused ja aiandussaadused,Maatalous- ja puutarhatuotteet,...,Žemės ūkio ir sodininkystės produktai,Lauksaimniecības un dārzkopības produkcija,Prodotti agrikoli u ta' l-ortikultura,Land- en tuinbouwproducten,Produkty rolnictwa i ogrodnictwa,Produtos agrícolas e hortofrutícolas,Produse agricole şi horticole,Poľnohospodárske a záhradnícke produkty,Kmetijski in hortikulturni proizvodi,Jordbruks- och trädgårdsprodukter
2,03110000-5,Продукти на растениевъдството; живи животни и ...,Produkty rostlinné výroby v zelinářství a zahr...,"Markafgrøder, gartneri- og havebrugsprodukter",Feldfrüchte und Erzeugnisse des Erwerbsgartenbaus,Φυτά και προϊόντα μεγάλης καλλιέργειας και κηπ...,"Crops, products of market gardening and hortic...","Cultivos, productos comerciales de jardinería ...","Põllukultuurid, puu- ja köögiviljad ning aiand...",Kauppapuutarhoista ja puutarhanhoidosta saadut...,...,"Žemės ūkio augalai, prekinės daržininkystės ir...","Augkopības, dārzkopības un puķkopības produkcija","Uċuħ tar-raba, prodotti mill-ġonna u l-oltikol...",Land- en tuinbouwgewassen en -producten,"Rośliny uprawne, produkty warzywnictwa i ogrod...","Cereais, produtos de culturas industriais e da...","Produse agricole, produse horticole şi de grăd...","Plodiny, produkty zeleninárstva a záhradníctva","Poljščine, proizvodi tržnega sadjarstva in vrt...","Grödor, trädgårdshandels- och trädgårdsprodukter"


9454 codes, une colonne par langue (EN, FR, etc.). Je cherche les codes dont le **libellé anglais**
contient des mots liés à mon métier : audit, security, testing, penetration.

In [20]:
mots = ["audit", "security", "testing", "penetration"]
masque_mot = cpv["EN"].str.lower().str.contains("|".join(mots), na=False)
print("Codes dont le libellé contient ces mots :", masque_mot.sum())
# aperçu rapide
for _, l in cpv[masque_mot].head(8).iterrows():
    print(f"  {str(l['CODE']).split('-')[0]:10} | {l['EN']}")

Codes dont le libellé contient ces mots : 84
  22450000   | Security-type printed matter
  33156000   | Psychology testing devices
  33696200   | Blood-testing reagents
  35000000   | Security, fire-fighting, police and defence equipment
  35100000   | Emergency and security equipment
  35120000   | Surveillance and security systems and devices
  35121000   | Security equipment
  35121300   | Security fittings


Attention : il y en a beaucoup (~85), mais avec du **bruit** : « Blood-testing reagents »,
« Convex security mirrors », « Cybercafé services »... Le mot « testing » ou « security » apparaît
dans des domaines qui n'ont rien à voir avec la cybersécurité. Je dois donc **trier**.

La cybersécurité informatique est surtout dans la **branche 72** (services informatiques) et quelques
codes **487** (logiciels de sécurité). Je combine donc : libellé pertinent **ET** code dans ces
branches.

In [21]:
cpv["code8"] = cpv["CODE"].str.split("-").str[0]
masque_info = cpv["code8"].str.startswith("72") | cpv["code8"].str.startswith("487")
pertinents = cpv[masque_mot & masque_info]

print("Codes CPV retenus (1re recherche) :\n")
for _, l in pertinents.iterrows():
    print(f"  {l['code8']:10} | {l['EN']}")

Codes CPV retenus (1re recherche) :

  48730000   | Security software package
  48731000   | File security software package
  48732000   | Data security software package
  72140000   | Computer hardware acceptance testing consultancy services
  72150000   | Computer audit consultancy and hardware consultancy services
  72212730   | Security software development services
  72212731   | File security software development services
  72212732   | Data security software development services
  72212984   | Program testing software development services
  72226000   | System software acceptance testing consultancy services
  72254000   | Software testing
  72254100   | Systems testing services
  72800000   | Computer audit and testing services
  72810000   | Computer audit services
  72820000   | Computer testing services


Une quinzaine de codes (audit, essai, logiciels de sécurité...). Mais je me demande si je n'ai
pas **raté** des codes en ne cherchant que 4 mots. La cybersécurité, c'est aussi l'antivirus, le
pare-feu, le chiffrement, la sauvegarde, la reprise après sinistre... J'élargis donc ma recherche
de mots dans la nomenclature, toujours dans les branches informatique/logiciel.

In [22]:
termes_larges = ["firewall", "intrusion", "encryption", "antivirus", "virus", "malware",
                 "network security", "information security", "cryptograph", "authentication",
                 "access control", "cyber", "data protection", "disaster recovery", "backup"]
masque_large = cpv["EN"].str.lower().str.contains("|".join(termes_larges), na=False)
nouveaux = cpv[masque_large & masque_info]

# ce que cette 2e recherche ajoute par rapport à la 1re
deja = set(pertinents["code8"])
ajouts = nouveaux[~nouveaux["code8"].isin(deja)]
print("Codes CPV spécifiques SUPPLÉMENTAIRES trouvés :\n")
for _, l in ajouts.iterrows():
    print(f"  {l['code8']:10} | {l['EN']}")

Codes CPV spécifiques SUPPLÉMENTAIRES trouvés :

  48710000   | Backup or recovery software package
  48760000   | Virus protection software package
  48761000   | Anti-virus software package
  72212710   | Backup or recovery software development services
  72212760   | Virus protection software development services
  72212761   | Anti-virus software development services
  72251000   | Disaster recovery services


La recherche élargie révèle des codes que j'avais ratés : antivirus (`48760000`, `48761000`),
sauvegarde (`48710000`), reprise après sinistre (`72251000`), et leurs services de développement.
Ce sont bien des codes **spécifiques** à la sécurité, donc je les ajoute à ma liste.

Ma liste finale combine les deux recherches.

In [23]:
CODES_CPV_CYBER = sorted(set(pertinents["code8"]) | set(nouveaux["code8"]))
print(f"Liste finale : {len(CODES_CPV_CYBER)} codes CPV spécifiques")
print(CODES_CPV_CYBER)

Liste finale : 22 codes CPV spécifiques
['48710000', '48730000', '48731000', '48732000', '48760000', '48761000', '72140000', '72150000', '72212710', '72212730', '72212731', '72212732', '72212760', '72212761', '72212984', '72226000', '72251000', '72254000', '72254100', '72800000', '72810000', '72820000']


**Une tentation à éviter :** pourquoi ne pas ajouter aussi les codes **génériques** que je vois
parfois sur des annonces cyber, comme `72000000` (services informatiques) ? Parce qu'un code
générique ne signifie pas « cyber » : il couvre TOUT l'informatique. Je le vérifie en comptant
combien d'annonces actives il représente.

In [24]:
# Combien d'annonces pour un code GÉNÉRIQUE vs nos codes SPÉCIFIQUES ?
print("72000000 (services informatiques, générique) :", total_annonces("classification-cpv=72000000"))
print("72810000 (audit informatique, spécifique)    :", total_annonces("classification-cpv=72810000"))

72000000 (services informatiques, générique) : 32568
72810000 (audit informatique, spécifique)    : 95


Le code générique `72000000` représente des **dizaines de milliers** d'annonces (tout
l'informatique : sites web, ERP, hébergement...), alors qu'un code spécifique comme `72810000` en
donne quelques centaines. Ajouter les codes génériques **noierait** mes vraies opportunités cyber
sous des milliers d'annonces hors sujet.

**Donc :** je garde uniquement les codes CPV **spécifiques**. Pour attraper les annonces cyber
classées sous un code générique (ex. un pentest rangé en `72000000`), je m'appuie sur les
**mots-clés** du titre, pas sur les codes génériques. Et le **scoring IA** écarte le bruit résiduel.
C'est la combinaison : CPV spécifiques + mots-clés + IA.

## 9. Les mots-clés : faut-il chercher dans d'autres langues ?

Les codes CPV captent les annonces bien classées. Pour les annonces cyber classées sous un code
**générique** (ex. un pentest rangé en `72000000`), je m'appuie sur des **mots-clés** dans le titre.
Mes mots-clés sont en anglais. Mais les titres TED existent dans 24 langues : est-ce que je rate les
annonces dont le titre n'est pas en anglais ? Je teste en comparant un mot anglais à ses équivalents
dans d'autres langues.

In [25]:
def total(q):
    body = {"query": q + " AND notice-type=cn-standard", "fields": ["publication-number"],
            "limit": 1, "scope": "ACTIVE"}
    return requests.post(URL, json=body, timeout=60).json()["totalNoticeCount"]

print("ANGLAIS  'vulnerability'  :", total('notice-title ~ "vulnerability"'))
print("FRANÇAIS 'vulnérabilité'  :", total('notice-title ~ "vulnérabilité"'))
print("ALLEMAND 'Sicherheit'     :", total('notice-title ~ "Sicherheit"'))
print("POLONAIS 'bezpieczeństwa' :", total('notice-title ~ "bezpieczeństwa"'))

ANGLAIS  'vulnerability'  : 63
FRANÇAIS 'vulnérabilité'  : 60
ALLEMAND 'Sicherheit'     : 2559
POLONAIS 'bezpieczeństwa' : 2411


Conclusion : chaque langue a ses propres annonces (l'allemand et le polonais en ont des
milliers !), donc mes mots anglais en ratent beaucoup. **Mais attention** : « Sicherheit » (allemand)
ou « bezpieczeństwa » (polonais) veulent dire « sécurité » au sens **large** (sécurité physique,
incendie...), pas seulement cyber. Si j'ajoute des mots aussi génériques, je ramène énormément de
bruit. Je dois donc choisir des termes **précis** par langue (l'équivalent de « cybersécurité »,
« test d'intrusion »), pas « sécurité » tout court. Je teste quelques termes précis.

In [26]:
for nom, q in {
    "FR cybersécurité":       'notice-title ~ "cybersécurité"',
    "DE Cybersicherheit":     'notice-title ~ "Cybersicherheit"',
    "DE Penetrationstest":    'notice-title ~ "Penetrationstest"',
    "PL cyberbezpieczeństwo": 'notice-title ~ "cyberbezpieczeństwo"',
    "ES ciberseguridad":      'notice-title ~ "ciberseguridad"',
}.items():
    print(f"  {nom:24} : {total(q)}")

  FR cybersécurité         : 48
  DE Cybersicherheit       : 2
  DE Penetrationstest      : 7
  PL cyberbezpieczeństwo   : 151
  ES ciberseguridad        : 30


Ces termes précis donnent des volumes raisonnables (quelques dizaines à ~150), sans l'explosion
de « Sicherheit ». Ils valent donc la peine d'être ajoutés. Je ne vise pas les 24 langues (ce serait
disproportionné, et les codes CPV couvrent déjà le principal indépendamment de la langue) : j'ajoute
les langues majeures (français, allemand, polonais, espagnol) avec des termes précis.

## 10. Ma requête de collecte cyber (CPV + mots-clés multilingues)

Je combine les codes CPV trouvés (signal fort) ET des mots-clés cyber en plusieurs langues, en me
limitant aux mises en concurrence (`notice-type=cn-standard`), c'est-à-dire des appels d'offres
ouverts (pas des avis d'attribution déjà clôturés).

In [27]:
filtre_cpv = " OR ".join(f"classification-cpv={c}" for c in CODES_CPV_CYBER)
filtre_mots = ('notice-title ~ "cybersecurity" OR notice-title ~ "penetration" '
               'OR notice-title ~ "ISO 27001" OR notice-title ~ "security audit" '
               'OR notice-title ~ "vulnerability" OR notice-title ~ "pentest" '
               'OR notice-title ~ "cybersécurité" OR notice-title ~ "cyberbezpieczeństwo" '
               'OR notice-title ~ "ciberseguridad" OR notice-title ~ "Cybersicherheit" '
               'OR notice-title ~ "Penetrationstest"')

REQUETE_CYBER = (f"({filtre_cpv} OR {filtre_mots}) "
                 "AND notice-type=cn-standard SORT BY publication-date DESC")

data = chercher(REQUETE_CYBER, limit=100)
print("Annonces cyber (mises en concurrence) :", data["totalNoticeCount"])
print("Récupérées dans l'échantillon :", len(data["notices"]))

Annonces cyber (mises en concurrence) : 1488
Récupérées dans l'échantillon : 100


## 11. Contrôle qualité des données

Avant de coder le collecteur, je vérifie la qualité sur cet échantillon réel.

In [28]:
notices = data["notices"]
n = len(notices)
print(f"Échantillon : {n} annonces\n")
print("a) Complétude :")
for champ in CHAMPS:
    presents = sum(1 for x in notices if x.get(champ) not in (None, "", [], {}))
    print(f"   {champ:20} : {presents}/{n}  ({100*presents//max(n,1)}%)")

Échantillon : 100 annonces

a) Complétude :
   publication-number   : 100/100  (100%)
   notice-title         : 100/100  (100%)
   buyer-name           : 100/100  (100%)
   buyer-country        : 100/100  (100%)
   publication-date     : 100/100  (100%)
   deadline             : 13/100  (13%)
   notice-type          : 100/100  (100%)
   classification-cpv   : 100/100  (100%)
   links                : 100/100  (100%)


In [29]:
print("b) Doublons sur publication-number :")
nums = [x.get("publication-number") for x in notices]
print(f"   {len(nums)} numéros, {len(set(nums))} uniques -> {len(nums)-len(set(nums))} doublon(s)")

b) Doublons sur publication-number :
   100 numéros, 100 uniques -> 0 doublon(s)


In [30]:
print("c) Langues des titres :")
sans_anglais = sum(1 for x in notices
                   if isinstance(x.get("notice-title"), dict) and "eng" not in x["notice-title"])
print(f"   annonces sans titre anglais : {sans_anglais}/{n}")

c) Langues des titres :
   annonces sans titre anglais : 0/100


In [31]:
print("d) Date limite (deadline) :")
def premiere(d): return d[0] if isinstance(d, list) else d
avec_dl = [x for x in notices if x.get("deadline")]
print(f"   présente dans : {len(avec_dl)}/{n}")
if avec_dl:
    print("   exemple de format :", premiere(avec_dl[0]["deadline"]))
    auj = date.today().isoformat()
    passees = sum(1 for x in avec_dl if str(premiere(x["deadline"]))[:10] < auj)
    print(f"   déjà passées alors que scope=ACTIVE : {passees}/{len(avec_dl)}")

d) Date limite (deadline) :
   présente dans : 13/100
   exemple de format : 2026-07-06T17:00:00+01:00
   déjà passées alors que scope=ACTIVE : 4/13


In [32]:
print("e) Lien : je le reconstruis depuis le numéro")
num = notices[0]["publication-number"]
print("   ", f"https://ted.europa.eu/fr/notice/{num}")

e) Lien : je le reconstruis depuis le numéro
    https://ted.europa.eu/fr/notice/424201-2026


### Ce que j'apprends

- Champs **fiables à 100 %** : `publication-number`, `notice-title`, `buyer-name`,
  `buyer-country`, `publication-date`, `classification-cpv`, `links`.
- **`publication-number` unique** (0 doublon) -> ma clé.
- **Titres toujours en anglais** -> je prends `eng`.
- **`deadline` souvent absente, et parfois déjà passée même en scope ACTIVE.** Je ne m'y fie donc
  pas seule : si elle est présente et dépassée, l'annonce est close ; sinon je me base sur `ACTIVE`.
  Format à normaliser : `2026-06-29T15:00:00+01:00` -> `2026-06-29`.
- **Liens** : structure complexe -> je reconstruis `https://ted.europa.eu/fr/notice/<num>`.

## 12. Les détails riches (prix, procédure, gagnant) sont-ils dans l'API ?

En cliquant sur une annonce du site, j'ai vu plein de détails : valeur estimée (le prix), type de
procédure, durée, gagnant, email de l'acheteur. Est-ce que l'API a ces champs ? Je cherche dans la
liste.

In [33]:
for label, ms in {"valeur/prix":["estimated-value","value-cur"], "procédure":["procedure-type"],
                  "durée":["contract-duration"], "gagnant":["winner"],
                  "description":["description-lot","description-proc"]}.items():
    trouves = []
    for m in ms:
        trouves += [c for c in champs if m in c.lower() and not c.startswith("BT-")]
    print(f"{label:14} -> {trouves[:4]}")

valeur/prix    -> ['framework-estimated-value-glo', 'group-framework-re-estimated-value-cur-res', 'estimated-value-cur-lot', 'framework-estimated-value-cur-glo']
procédure      -> ['procedure-type']
durée          -> ['contract-duration-period-oth-part', 'contract-duration-start-date-part', 'contract-duration-end-date-part', 'contract-duration-period-oth-lot']
gagnant        -> ['winner-post-code', 'winner-country', 'winner-touchpoint-post-code', 'winner-gateway']
description    -> ['option-description-lot', 'missing-info-submission-description-lot', 'selection-criterion-description-lot', 'renewal-description-lot']


Oui, l'API contient ces détails. Donc **pas besoin de scraper la page de détail** : tout est
dans l'API, il suffit d'ajouter les champs voulus à ma requête.

**Stratégie (la même que pour la BCEAO) :** je ne récupère pas tout pour tout le monde.
- À la collecte : champs **essentiels** (titre, organisme, pays, date, lien, CPV) pour toutes les
  annonces ouvertes.
- Après le scoring : champs **riches** (prix, procédure, durée, description, documents) seulement
  pour les annonces jugées **pertinentes**, en enrichissant la requête API.

Remarque : un avis avec une section « Results / Winner » est un **avis d'attribution** (marché déjà
gagné), pas une opportunité. Le filtre `notice-type=cn-standard` les écarte.

## 13. Décisions de conception pour `collecter_ted()`

Pour alimenter la même base que la BCEAO, je produis le même format :

| Champ de ma base | Source TED |
|---|---|
| `cle_unique` | `ted::<publication-number>` |
| `intitule` | `notice-title["eng"]` |
| `source` | `"TED"` |
| organisme / pays | `buyer-name` / `buyer-country` |
| `date_publication` | `publication-date` normalisée |
| `delai_soumission` | `deadline` si présent, normalisé, sinon `null` |
| `lien` | `https://ted.europa.eu/fr/notice/<numéro>` |
| `statut_source` | `"en cours"` (ACTIVE), ou `"clôturé"` si date limite dépassée |

Le prix et les détails riches seront ajoutés **plus tard**, pour les annonces pertinentes.

## 14. Prochaines étapes

1. Écrire `collecter_ted()` avec les champs **essentiels**, testée sur de vraies données.
2. L'intégrer dans `collecte_et_scoring.py`, à côté de `collecter_bceao()`.
3. Vérifier que le scoring repère les vraies annonces cyber.
4. Plus tard : ajouter le prix et les détails pour les pertinentes ; gérer la pagination
   (`iterationNextToken`) si besoin de plus de 100 annonces.